# Retrieval Diversity and Redundancy in RAG
A clean end-to-end notebook for studying relevance, redundancy, and coverage in retrieval for RAG.

This version rewrites the full workflow from scratch, starting with:
1. downloading the 2019 and 2020 TREC Deep Learning query sets from MS MARCO,
2. inspecting what a query and graded relevance judgments look like,
3. building a frozen evaluation set,
4. constructing a judged-document candidate pool,
5. building dense and sparse retrieval representations,
6. implementing baseline and diversity-aware retrieval methods,
7. evaluating both ranking quality and coverage efficiency.

The notebook is organized so that another researcher can read it top to bottom and understand both the experimental setup and the code.

## 0. Research framing

**Goal.** Retrieve a set of documents that covers the meaning of a query while minimizing redundancy.

**Main question.** Can diversity-aware retrieval methods select a smaller, less redundant set of documents than standard retrieval baselines without losing too much useful information?

**Methods implemented in this notebook**
- Dense retrieval
- Sparse retrieval using standard BM25
- Dense-sparse score fusion
- Hybrid pricing
- Hybrid MMR
- Dual residual tracking

**Important design decision for dual residual in this rewrite**
- The sparse side stays close to standard sparse retrieval.
- We do **not** replace sparse retrieval with raw token-overlap coverage scoring.
- Instead, dual residual uses a **weighted BM25-style query**, where query term weights decay over time as their lexical demand is covered.
- Stopping is based on a **combined dense + sparse residual exhaustion signal**, not only a sparse threshold.

## 1. Optional installs

Uncomment and run if the environment does not already have the required libraries.

In [7]:
%pip install -q ir_datasets sentence-transformers rank_bm25 scikit-learn pandas matplotlib tqdm

Note: you may need to restart the kernel to use updated packages.


## 2. Imports

In [8]:
import json
import math
import os
import pickle
import glob
import random
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from itertools import product
from typing import Dict, List, Tuple, Iterable, Optional

import ir_datasets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

## 3. Configuration

In [9]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATASETS = [
    "msmarco-passage/trec-dl-2019",
    "msmarco-passage/trec-dl-2020",
]

CACHE_DIR = "rag_diversity_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 10

# Retrieval defaults
DEFAULT_ALPHA = 0.75
DEFAULT_LAMBDA = 0.30
DEFAULT_ETA = 0.35

# Dual residual defaults
DEFAULT_BETA_STOP = 0.50
DEFAULT_MIN_COMBINED_RESIDUAL_FRAC = 0.10
DEFAULT_MIN_COMBINED_SCORE = 0.10

# Coverage thresholds for efficiency metrics
SEMANTIC_THRESHOLDS = [0.80, 0.90, 0.95]
SPARSE_THRESHOLDS = [0.80, 0.90, 0.95]

# Candidate pool controls
MAX_CANDIDATE_DOCS = None   # set to an integer if you want to subsample for faster experiments

# Export controls
RESULTS_CSV = os.path.join(CACHE_DIR, "method_results.csv")
SWEEP_CSV = os.path.join(CACHE_DIR, "dual_residual_sweep.csv")

print("Cache directory:", CACHE_DIR)

Cache directory: rag_diversity_cache


## 4. Load the 2019 and 2020 TREC-DL datasets

These datasets give:
- queries
- qrels with graded relevance labels
- access to the MS MARCO passage store through `docs_store()`

In [10]:
datasets = [ir_datasets.load(name) for name in DATASETS]
datasets

[Dataset(id='msmarco-passage/trec-dl-2019', provides=['docs', 'queries', 'qrels', 'scoreddocs']),
 Dataset(id='msmarco-passage/trec-dl-2020', provides=['docs', 'queries', 'qrels', 'scoreddocs'])]

## 5. Inspect what a query and graded judgments look like

This is the first sanity check in the notebook.

In [11]:
def load_queries_and_qrels(dataset):
    queries = {q.query_id: q.text for q in dataset.queries_iter()}
    qrels = defaultdict(dict)
    for qr in dataset.qrels_iter():
        qrels[qr.query_id][qr.doc_id] = qr.relevance
    return queries, dict(qrels)

all_queries = {}
all_qrels = {}
for ds in datasets:
    q, r = load_queries_and_qrels(ds)
    all_queries.update(q)
    all_qrels.update(r)

sample_qid = sorted(all_qrels.keys())[0]
print("Sample query id:", sample_qid)
print("Sample query text:", all_queries[sample_qid])
print("\nSample qrels for this query:")
print(all_qrels[sample_qid])

[INFO] Please confirm you agree to the MSMARCO data usage agreement found at <http://www.msmarco.org/dataset.aspx>
[INFO] [starting] https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2019-queries.tsv.gz
[INFO] [finished] https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2019-queries.tsv.gz: [00:00] [4.28kB] [13.5MB/s]
[INFO] [starting] https://trec.nist.gov/data/deep/2019qrels-pass.txt                                               
[INFO] [finished] https://trec.nist.gov/data/deep/2019qrels-pass.txt: [00:00] [187kB] [3.21MB/s]
[INFO] [starting] https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2020-queries.tsv.gz
[INFO] [finished] https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2020-queries.tsv.gz: [00:00] [4.13kB] [22.0MB/s]
[INFO] [starting] https://trec.nist.gov/data/deep/2020qrels-pass.txt                                               
[INFO] [finished] https://trec.nist.gov/data/deep/2020qrels-pass.txt: [00:

Sample query id: 1030303
Sample query text: who is aziz hashim

Sample qrels for this query:
{'1038342': 0, '1042969': 0, '1044043': 0, '1056584': 0, '1070133': 0, '1070134': 0, '1086238': 0, '1104408': 0, '1161432': 0, '1161439': 0, '1166176': 0, '1172906': 0, '1207443': 0, '1227100': 0, '1269099': 0, '1292821': 0, '1292822': 0, '1305520': 0, '1305521': 0, '1305528': 0, '1358683': 0, '1369019': 0, '1376556': 0, '1376558': 0, '1397758': 0, '1402779': 0, '1451846': 0, '1451848': 0, '1451850': 0, '1451851': 0, '1462522': 0, '1492373': 0, '1538148': 0, '1538153': 0, '163871': 0, '1666520': 0, '1777312': 0, '1784767': 0, '1811800': 0, '1813557': 0, '1813561': 0, '1905988': 0, '196026': 0, '2001450': 0, '2057929': 0, '2146415': 0, '2194939': 0, '2205078': 0, '2217415': 0, '2217635': 0, '22484': 0, '226464': 0, '2484376': 0, '2488127': 0, '2563261': 0, '2597066': 0, '2620119': 0, '2628843': 0, '2628844': 0, '2712400': 0, '2740876': 0, '2782702': 0, '288491': 0, '292060': 0, '309441': 0, '310

If you want to inspect a random query with graded labels, run the next cell a few times.

In [12]:
import random

random_qid = random.choice(list(all_qrels.keys()))
print("Random query id:", random_qid)
print("Query:", all_queries[random_qid])
print("Number of judged docs:", len(all_qrels[random_qid]))
print("Top judged docs by grade:")
for doc_id, rel in sorted(all_qrels[random_qid].items(), key=lambda x: (-x[1], x[0]))[:10]:
    print(doc_id, "->", rel)

Random query id: 1106979
Query: define pareto chart in statistics
Number of judged docs: 157
Top judged docs by grade:
3290649 -> 3
8190140 -> 3
1039252 -> 2
2306620 -> 2
2375312 -> 2
2884219 -> 2
3290646 -> 2
3290651 -> 2
348054 -> 2
348057 -> 2


## 6. Build the frozen evaluation set

We keep only queries with at least one relevant document.
That gives a stable query set for consistent benchmarking.

In [13]:
def build_frozen_query_set(queries: Dict[str, str], qrels: Dict[str, Dict[str, int]]):
    queries_frozen = {}
    qrels_frozen = {}
    for qid, text in queries.items():
        judged = qrels.get(qid, {})
        if any(rel > 0 for rel in judged.values()):
            queries_frozen[qid] = text
            qrels_frozen[qid] = judged
    return queries_frozen, qrels_frozen

queries_frozen, qrels_frozen = build_frozen_query_set(all_queries, all_qrels)
print("Frozen query count:", len(queries_frozen))
print("Example query id:", next(iter(queries_frozen)))

Frozen query count: 97
Example query id: 156493


## 7. Build the judged-document candidate pool

To keep experiments controlled and reproducible, we take the union of all judged documents
appearing in the frozen qrels, then fetch the passage text for those docs.

In [ ]:
docs_store = datasets[0].docs_store()

candidate_doc_ids = sorted({doc_id for qid in qrels_frozen for doc_id in qrels_frozen[qid]})
if MAX_CANDIDATE_DOCS is not None:
    candidate_doc_ids = candidate_doc_ids[:MAX_CANDIDATE_DOCS]

doc_text = {}
missing_doc_ids = []

for doc_id in tqdm(candidate_doc_ids, desc="Fetching judged passages"):
    doc = docs_store.get(doc_id)
    if doc is None:
        missing_doc_ids.append(doc_id)
        continue
    text = getattr(doc, "text", None)
    if text is None:
        text = str(doc)
    doc_text[doc_id] = text

candidate_doc_ids = [d for d in candidate_doc_ids if d in doc_text]
doc_index = {doc_id: i for i, doc_id in enumerate(candidate_doc_ids)}

print("Candidate docs loaded:", len(candidate_doc_ids))
print("Missing docs:", len(missing_doc_ids))

Fetching judged passages:   0%|          | 0/20349 [00:00<?, ?it/s]

[INFO] [starting] building docstore

docs_iter:   0%|                                   | 0/8841823 [00:00<?, ?doc/s]
[INFO] If you have a local copy of https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz, you can symlink it here to avoid downloading it again: /home/users/jacobr1/.ir_datasets/downloads/31644046b18952c1386cd4564ba2ae69

docs_iter:   0%|                                   | 0/8841823 [00:00<?, ?doc/s]
[INFO] [starting] https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz

docs_iter:   0%|                                   | 0/8841823 [00:00<?, ?doc/s]

https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz: 0.0%| 0.00/1.06G [00:00<?, ?B/s]

https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz: 0.1%| 918k/1.06G [00:00<02:18, 7.65MB/s]

https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz: 0.4%| 4.46M/1.06G [00:00<00:52, 2

## 8. Inspect a judged document example

In [ ]:
sample_doc_id = next(iter(doc_text))
print("Sample doc id:", sample_doc_id)
print(doc_text[sample_doc_id][:1000])

## 9. Tokenization helpers

In [ ]:
TOKEN_PATTERN = re.compile(r"[a-z0-9]+")

def simple_tokenize(text: str) -> List[str]:
    return TOKEN_PATTERN.findall(text.lower())

tokenized_docs = [simple_tokenize(doc_text[doc_id]) for doc_id in candidate_doc_ids]
doc_term_freqs = [Counter(tokens) for tokens in tokenized_docs]
doc_lengths = np.array([len(tokens) for tokens in tokenized_docs], dtype=float)
avg_doc_len = float(doc_lengths.mean()) if len(doc_lengths) else 0.0

print("Average tokenized document length:", round(avg_doc_len, 2))

## 10. Standard sparse retrieval: BM25

This is the normal sparse baseline.

We will also reuse BM25 components inside dual residual, but with **residual query weights**.
The document side remains standard BM25-style term-frequency scoring.

In [ ]:
bm25 = BM25Okapi(tokenized_docs)
print("BM25 index built over", len(tokenized_docs), "documents")

## 11. Dense representation

We encode all candidate documents once, normalize them, and keep them in memory.

In [ ]:
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

doc_embedding_cache = os.path.join(CACHE_DIR, "doc_embeddings.npy")
if os.path.exists(doc_embedding_cache):
    doc_embeddings = np.load(doc_embedding_cache)
else:
    texts = [doc_text[doc_id] for doc_id in candidate_doc_ids]
    doc_embeddings = embed_model.encode(
        texts,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    np.save(doc_embedding_cache, doc_embeddings)

print("Doc embeddings shape:", doc_embeddings.shape)

## 12. Query embedding helper

In [ ]:
def embed_query(query_text: str) -> np.ndarray:
    return embed_model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0]

## 13. Core scoring helpers

In [ ]:
def normalize_scores(scores: np.ndarray) -> np.ndarray:
    scores = np.asarray(scores, dtype=float)
    if len(scores) == 0:
        return scores
    lo = np.min(scores)
    hi = np.max(scores)
    if hi - lo < 1e-12:
        return np.zeros_like(scores)
    return (scores - lo) / (hi - lo)

def dense_scores_for_query(query_text: str) -> np.ndarray:
    q_emb = embed_query(query_text)
    return np.asarray(doc_embeddings @ q_emb, dtype=float)

def sparse_scores_for_query(query_text: str) -> np.ndarray:
    q_tokens = simple_tokenize(query_text)
    return np.array(bm25.get_scores(q_tokens), dtype=float)

def get_relevance_grade(qid: str, doc_id: str) -> int:
    return int(qrels_frozen.get(qid, {}).get(doc_id, 0))

def upper_triangle_mean(sim_matrix: np.ndarray) -> float:
    n = sim_matrix.shape[0]
    if n < 2:
        return 0.0
    iu = np.triu_indices(n, k=1)
    return float(sim_matrix[iu].mean())

## 14. Weighted BM25 for dual residual

This is the key change.

Instead of replacing sparse retrieval with simple token overlap, we keep the sparse side close to normal BM25.
The difference is that the query now has **residual token weights**.

For a weighted query:
- each query token contributes according to its remaining residual weight
- document scoring still uses BM25-style tf, idf, and document-length normalization

In [ ]:
# Extract BM25 internals
BM25_K1 = bm25.k1
BM25_B = bm25.b
BM25_IDF = bm25.idf

def weighted_bm25_scores_from_residual(sparse_residual: Counter) -> np.ndarray:
    scores = np.zeros(len(candidate_doc_ids), dtype=float)
    if not sparse_residual:
        return scores

    for tok, q_weight in sparse_residual.items():
        idf = BM25_IDF.get(tok, 0.0)
        if abs(idf) < 1e-12:
            continue

        for i, tf_counter in enumerate(doc_term_freqs):
            tf = tf_counter.get(tok, 0)
            if tf == 0:
                continue
            denom = tf + BM25_K1 * (1.0 - BM25_B + BM25_B * (doc_lengths[i] / avg_doc_len))
            scores[i] += q_weight * idf * (tf * (BM25_K1 + 1.0) / denom)

    return scores

def build_sparse_query_weights(query_text: str) -> Counter:
    return Counter(simple_tokenize(query_text))

def update_sparse_residual(doc_text_value: str, sparse_residual: Counter, decay: float = 1.0) -> Counter:
    doc_tokens = set(simple_tokenize(doc_text_value))
    new_residual = Counter(sparse_residual)
    for tok in list(new_residual.keys()):
        if tok in doc_tokens:
            new_residual[tok] = max(0.0, new_residual[tok] - decay)
            if new_residual[tok] <= 0:
                del new_residual[tok]
    return new_residual

## 15. Retrieval methods

In [ ]:
def retrieve_dense(query_text: str, k: int = TOP_K) -> List[str]:
    scores = dense_scores_for_query(query_text)
    top_idx = np.argsort(-scores)[:k]
    return [candidate_doc_ids[i] for i in top_idx]

def retrieve_sparse(query_text: str, k: int = TOP_K) -> List[str]:
    scores = sparse_scores_for_query(query_text)
    top_idx = np.argsort(-scores)[:k]
    return [candidate_doc_ids[i] for i in top_idx]

def retrieve_fusion(query_text: str, alpha: float = DEFAULT_ALPHA, k: int = TOP_K) -> List[str]:
    dense_scores = normalize_scores(dense_scores_for_query(query_text))
    sparse_scores = normalize_scores(sparse_scores_for_query(query_text))
    fused = alpha * dense_scores + (1.0 - alpha) * sparse_scores
    top_idx = np.argsort(-fused)[:k]
    return [candidate_doc_ids[i] for i in top_idx]

def retrieve_hybrid_mmr(
    query_text: str,
    alpha: float = DEFAULT_ALPHA,
    lambda_div: float = DEFAULT_LAMBDA,
    k: int = TOP_K,
) -> List[str]:
    #compute dense, sparse, and fusion scores as a starter point for each doc
    dense_scores = normalize_scores(dense_scores_for_query(query_text))
    sparse_scores = normalize_scores(sparse_scores_for_query(query_text))
    relevance = alpha * dense_scores + (1.0 - alpha) * sparse_scores

    selected = []
    remaining = set(range(len(candidate_doc_ids)))

    #iteratively begin/continnue to choose documents
    while remaining and len(selected) < k:
        
        best_i = None
        best_score = -1e18
        for i in remaining:
            #if nothing selected, just take highest score
            if not selected:
                score = relevance[i]
            else:
                #else, compare remaining docs to selected docus and recompute scores
                ##penalize documents for being too similar to an already chosen document
                #max takes highest similarity
                sim_to_selected = np.max(doc_embeddings[selected] @ doc_embeddings[i])
                score = relevance[i] - lambda_div * float(sim_to_selected)
            #keep track of best candidate, this is what takes a long time
            if score > best_score:
                best_score = score
                best_i = i
        #add best document to selected, remove from remaining
        selected.append(best_i)
        remaining.remove(best_i)

    return [candidate_doc_ids[i] for i in selected]

def retrieve_hybrid_pricing(
    query_text: str,
    alpha: float = DEFAULT_ALPHA,
    lambda_div: float = DEFAULT_LAMBDA,
    k: int = TOP_K,
) -> List[str]:
    dense_scores = normalize_scores(dense_scores_for_query(query_text))
    sparse_scores = normalize_scores(sparse_scores_for_query(query_text))
    relevance = alpha * dense_scores + (1.0 - alpha) * sparse_scores

    selected = []
    remaining = set(range(len(candidate_doc_ids)))

    while remaining and len(selected) < k:
        best_i = None
        best_score = -1e18
        for i in remaining:
            redundancy = 0.0
            if selected:
                redundancy = float(np.mean(doc_embeddings[selected] @ doc_embeddings[i]))
            score = relevance[i] - lambda_div * redundancy
            if score > best_score:
                best_score = score
                best_i = i
        selected.append(best_i)
        remaining.remove(best_i)

    return [candidate_doc_ids[i] for i in selected]

## 16. Dual residual tracking

This is the main method rewrite.

### Dense side
We maintain:
- a dense residual direction `q_dense_dir`
- a scalar `dense_mass_remaining`

After selecting a document with embedding `d`, we:
1. measure how much of the current semantic direction it covers,
2. subtract that covered direction,
3. renormalize the direction,
4. separately decay the scalar semantic mass so we still have an interpretable stopping signal.

### Sparse side
We maintain residual token weights over the query.
The sparse score is a **weighted BM25 score**, not a raw token-overlap count.

### Stopping
We compute:
- `dense_residual_frac`
- `sparse_residual_frac`

Then combine them:

`combined_residual_frac = beta_stop * dense_residual_frac + (1 - beta_stop) * sparse_residual_frac`

We stop when:
- combined residual demand is small enough, or
- the best remaining document under the residual objective is too weak, or
- we reach `k`

In [ ]:
def retrieve_dual_residual(
    query_text: str,
    alpha: float = DEFAULT_ALPHA,
    eta: float = DEFAULT_ETA,
    sparse_decay: float = 1.0,
    beta_stop: float = DEFAULT_BETA_STOP,
    min_combined_residual_frac: float = DEFAULT_MIN_COMBINED_RESIDUAL_FRAC,
    min_combined_score: Optional[float] = DEFAULT_MIN_COMBINED_SCORE,
    k: int = TOP_K,
    return_debug: bool = False,
):

    #get dense embedding (function edfined above)
    q_dense_dir = embed_query(query_text).copy()
    dense_mass_remaining = 1.0

    #intitalize sparse weights, once again with function before fom sparse retrieval    
    sparse_residual = build_sparse_query_weights(query_text)
    initial_sparse_mass = float(sum(sparse_residual.values())) if sparse_residual else 1.0

    #to keep track of documents been selected, documents remaining
    selected = []
    remaining = set(range(len(candidate_doc_ids)))
    history = []

    #so this is retrieval step, while documents are still left to be retrieved
    while remaining and len(selected) < k:
        #compute dense scores for docs using current dense vector
        dense_scores = doc_embeddings @ q_dense_dir
        dense_scores = normalize_scores(dense_scores)
        #computre sparse scores using current sparse vector
        sparse_scores = weighted_bm25_scores_from_residual(sparse_residual)
        sparse_scores = normalize_scores(sparse_scores)

        #find best remaining doc using scores
        combined = alpha * dense_scores + (1.0 - alpha) * sparse_scores

        remaining_list = list(remaining)
        best_i = remaining_list[int(np.argmax(combined[remaining_list]))]
        best_score = float(combined[best_i])

        #check sprase and dense vectors and see if we ahve done enough to stop
        sparse_mass_remaining = float(sum(sparse_residual.values()))
        sparse_residual_frac = sparse_mass_remaining / initial_sparse_mass if initial_sparse_mass > 0 else 0.0
        dense_residual_frac = dense_mass_remaining
        combined_residual_frac = beta_stop * dense_residual_frac + (1.0 - beta_stop) * sparse_residual_frac
        #check for stopping
        if len(selected) > 0:
            if combined_residual_frac < min_combined_residual_frac:
                break
            if min_combined_score is not None and best_score < min_combined_score:
                break

        selected.append(best_i)
        remaining.remove(best_i)

        #update dense residual term
        d = doc_embeddings[best_i]
        coverage = max(0.0, float(q_dense_dir @ d))

        q_dense_dir = q_dense_dir - eta * coverage * d
        norm = np.linalg.norm(q_dense_dir)
        if norm > 1e-12:
            q_dense_dir = q_dense_dir / norm

        dense_mass_remaining = max(0.0, dense_mass_remaining * (1.0 - eta * coverage))

        #update sparse residual term
        sparse_residual = update_sparse_residual(
            doc_text[candidate_doc_ids[best_i]],
            sparse_residual,
            decay=sparse_decay,
        )
        #for checkpointing
        history.append({
            "step": len(selected),
            "doc_id": candidate_doc_ids[best_i],
            "best_score": best_score,
            "dense_residual_frac": dense_mass_remaining,
            "sparse_residual_frac": float(sum(sparse_residual.values())) / initial_sparse_mass if initial_sparse_mass > 0 else 0.0,
        })

    selected_doc_ids = [candidate_doc_ids[i] for i in selected]

    if return_debug:
        return selected_doc_ids, history
    return selected_doc_ids

## 17. Evaluation metrics

We evaluate both ranking-style and coverage-style metrics.

In [ ]:
def dcg_at_k(retrieved_doc_ids: List[str], qid: str, k: int = TOP_K) -> float:
    score = 0.0
    for rank, doc_id in enumerate(retrieved_doc_ids[:k], start=1):
        rel = get_relevance_grade(qid, doc_id)
        gain = (2 ** rel) - 1
        discount = 1.0 / math.log2(rank + 1)
        score += gain * discount
    return score

def idcg_at_k(qid: str, k: int = TOP_K) -> float:
    rels = sorted(qrels_frozen[qid].values(), reverse=True)[:k]
    score = 0.0
    for rank, rel in enumerate(rels, start=1):
        gain = (2 ** rel) - 1
        discount = 1.0 / math.log2(rank + 1)
        score += gain * discount
    return score

def ndcg_at_k(retrieved_doc_ids: List[str], qid: str, k: int = TOP_K) -> float:
    denom = idcg_at_k(qid, k)
    if denom == 0:
        return 0.0
    return dcg_at_k(retrieved_doc_ids, qid, k) / denom

def irrelevant_at_k(retrieved_doc_ids: List[str], qid: str, k: int = TOP_K) -> int:
    return sum(1 for doc_id in retrieved_doc_ids[:k] if get_relevance_grade(qid, doc_id) <= 0)

def hit_grade3(retrieved_doc_ids: List[str], qid: str, k: int = TOP_K) -> int:
    return int(any(get_relevance_grade(qid, doc_id) == 3 for doc_id in retrieved_doc_ids[:k]))

def redundancy_at_k(retrieved_doc_ids: List[str], k: int = TOP_K) -> float:
    ids = retrieved_doc_ids[:k]
    if len(ids) < 2:
        return 0.0
    emb = np.vstack([doc_embeddings[doc_index[d]] for d in ids])
    sim = emb @ emb.T
    return upper_triangle_mean(sim)

## 18. Coverage metrics

The current judged candidate pool makes pure coverage difficult to define perfectly, but we can still build useful proxies.

### Semantic coverage
For each relevant judged document for a query, compute its maximum cosine similarity to the selected set.
Then average those maxima, weighting by relevance grade.

This measures how well the selected set semantically covers the judged relevant set.

### Sparse coverage
Take the original query token multiset.
Measure the fraction of query token mass that appears in at least one selected document, counting repeated tokens using the original query counts.

### Efficiency
How many retrieved documents are needed to exceed given semantic or sparse coverage thresholds?

In [ ]:
def relevant_doc_ids_for_query(qid: str) -> List[str]:
    return [doc_id for doc_id, rel in qrels_frozen[qid].items() if rel > 0 and doc_id in doc_index]

def semantic_coverage(retrieved_doc_ids: List[str], qid: str, k: int = TOP_K) -> float:
    selected = [doc_id for doc_id in retrieved_doc_ids[:k] if doc_id in doc_index]
    relevant = relevant_doc_ids_for_query(qid)

    if not selected or not relevant:
        return 0.0

    selected_emb = np.vstack([doc_embeddings[doc_index[d]] for d in selected])
    relevant_emb = np.vstack([doc_embeddings[doc_index[d]] for d in relevant])

    sim = relevant_emb @ selected_emb.T
    max_sim = sim.max(axis=1)

    weights = np.array([qrels_frozen[qid][d] for d in relevant], dtype=float)
    if weights.sum() <= 0:
        return float(max_sim.mean())
    return float(np.average(max_sim, weights=weights))

def sparse_query_coverage(retrieved_doc_ids: List[str], query_text: str, k: int = TOP_K) -> float:
    query_weights = build_sparse_query_weights(query_text)
    if not query_weights:
        return 0.0

    covered = Counter()
    for doc_id in retrieved_doc_ids[:k]:
        doc_tokens = set(simple_tokenize(doc_text[doc_id]))
        for tok in query_weights:
            if tok in doc_tokens:
                covered[tok] = query_weights[tok]

    covered_mass = float(sum(covered.values()))
    total_mass = float(sum(query_weights.values()))
    return covered_mass / total_mass if total_mass > 0 else 0.0

def docs_needed_for_semantic_coverage(retrieved_doc_ids: List[str], qid: str, threshold: float) -> Optional[int]:
    for j in range(1, len(retrieved_doc_ids) + 1):
        if semantic_coverage(retrieved_doc_ids[:j], qid, k=j) >= threshold:
            return j
    return None

def docs_needed_for_sparse_coverage(retrieved_doc_ids: List[str], query_text: str, threshold: float) -> Optional[int]:
    for j in range(1, len(retrieved_doc_ids) + 1):
        if sparse_query_coverage(retrieved_doc_ids[:j], query_text, k=j) >= threshold:
            return j
    return None

## 19. Unified evaluation wrapper

In [ ]:
import os

CHECKPOINT_DIR = "rag_diversity_cache/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

CACHE_DIR = "rag_diversity_cache"
CHECKPOINT_DIR = os.path.join(CACHE_DIR, "checkpoints")

RESULTS_CSV = os.path.join(CACHE_DIR, "method_results.csv")
RESULTS_ALL_METHODS_CSV = os.path.join(CACHE_DIR, "all_methods_results.csv")

CHECKPOINT_MANIFEST = os.path.join(CHECKPOINT_DIR, "checkpoint_manifest.json")

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
def evaluate_single_query(qid: str, method_name: str, retrieved_doc_ids: List[str], query_text: str, k: int = TOP_K) -> Dict:
    row = {
        "qid": qid,
        "method": method_name,
        "k": k,
        "nDCG@k": ndcg_at_k(retrieved_doc_ids, qid, k),
        "irrelevant@k": irrelevant_at_k(retrieved_doc_ids, qid, k),
        "redundancy@k": redundancy_at_k(retrieved_doc_ids, k),
        "hit_grade3@k": hit_grade3(retrieved_doc_ids, qid, k),
        "semantic_coverage@k": semantic_coverage(retrieved_doc_ids, qid, k),
        "sparse_coverage@k": sparse_query_coverage(retrieved_doc_ids, query_text, k),
        "docs_retrieved": len(retrieved_doc_ids[:k]),
    }
    for thr in SEMANTIC_THRESHOLDS:
        row[f"docs_to_semantic_{thr:.2f}"] = docs_needed_for_semantic_coverage(retrieved_doc_ids[:k], qid, thr)
    for thr in SPARSE_THRESHOLDS:
        row[f"docs_to_sparse_{thr:.2f}"] = docs_needed_for_sparse_coverage(retrieved_doc_ids[:k], query_text, thr)
    return row

def evaluate_method_over_queries(method_name: str, retrieve_fn, qids: Optional[List[str]] = None, k: int = TOP_K, verbose: bool = True) -> pd.DataFrame:
    if qids is None:
        qids = list(queries_frozen.keys())

    rows = []
    iterator = tqdm(qids, desc=f"Evaluating {method_name}") if verbose else qids
    for qid in iterator:
        query_text = queries_frozen[qid]
        retrieved_doc_ids = retrieve_fn(query_text, k=k)
        rows.append(evaluate_single_query(qid, method_name, retrieved_doc_ids, query_text, k=k))
    return pd.DataFrame(rows)

def safe_method_filename(method_name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", method_name)

def method_checkpoint_path(method_name: str, checkpoint_dir: str = CHECKPOINT_DIR) -> str:
    return os.path.join(checkpoint_dir, f"{safe_method_filename(method_name)}.csv")

def atomic_to_csv(df: pd.DataFrame, path: str) -> None:
    tmp_path = f"{path}.tmp"
    df.to_csv(tmp_path, index=False)
    os.replace(tmp_path, path)

def evaluate_method_over_queries_checkpointed(
    method_name: str,
    retrieve_fn,
    qids: Optional[List[str]] = None,
    k: int = TOP_K,
    verbose: bool = True,
    family: Optional[str] = None,
    params: Optional[Dict] = None,
    checkpoint_dir: str = CHECKPOINT_DIR,
    flush_every: int = 1,
    force_restart_method: bool = False,
):
    if qids is None:
        qids = list(queries_frozen.keys())
    params = params or {}
    os.makedirs(checkpoint_dir, exist_ok=True)

    ckpt_path = method_checkpoint_path(method_name, checkpoint_dir=checkpoint_dir)

    existing_df = None
    existing_qids = set()

    if (not force_restart_method) and os.path.exists(ckpt_path):
        existing_df = pd.read_csv(ckpt_path)
        if "qid" in existing_df.columns:
            existing_qids = set(existing_df["qid"].astype(str))

    qids = [str(qid) for qid in qids]
    remaining_qids = [qid for qid in qids if qid not in existing_qids]

    if verbose:
        print(f"{method_name}: {len(existing_qids)} done, {len(remaining_qids)} remaining")

    if len(remaining_qids) == 0:
        if existing_df is None:
            existing_df = pd.DataFrame()
        return existing_df, False

    new_rows = []
    wrote_anything = False
    iterator = tqdm(remaining_qids, desc=f"Evaluating {method_name}") if verbose else remaining_qids

    for idx, qid in enumerate(iterator, start=1):
        query_text = queries_frozen[qid]
        retrieved_doc_ids = retrieve_fn(query_text, k=k)

        row = evaluate_single_query(qid, method_name, retrieved_doc_ids, query_text, k=k)
        row["family"] = family
        for key, value in params.items():
            row[key] = value
        new_rows.append(row)

        should_flush = (
            flush_every is not None
            and flush_every > 0
            and (idx % flush_every == 0)
        )

        if should_flush:
            flush_df = pd.DataFrame(new_rows)

            if existing_df is not None and len(existing_df):
                combined_df = pd.concat([existing_df, flush_df], ignore_index=True)
            else:
                combined_df = flush_df

            atomic_to_csv(combined_df, ckpt_path)
            existing_df = combined_df
            new_rows = []
            wrote_anything = True

    if new_rows:
        flush_df = pd.DataFrame(new_rows)
        if existing_df is not None and len(existing_df):
            combined_df = pd.concat([existing_df, flush_df], ignore_index=True)
        else:
            combined_df = flush_df

        atomic_to_csv(combined_df, ckpt_path)
        existing_df = combined_df
        wrote_anything = True

    return existing_df, wrote_anything

## 20. Quick spot-check on one query

In [ ]:
test_qid = random.choice(list(queries_frozen.keys()))
test_query = queries_frozen[test_qid]
print("Test qid:", test_qid)
print("Query:", test_query)

for name, fn in [
    ("dense", retrieve_dense),
    ("sparse_bm25", retrieve_sparse),
    ("fusion", lambda q, k=TOP_K: retrieve_fusion(q, alpha=0.75, k=k)),
    ("dual_residual", lambda q, k=TOP_K: retrieve_dual_residual(q, alpha=0.75, eta=0.35, sparse_decay=1.0, beta_stop=0.5, min_combined_residual_frac=0.10, min_combined_score=0.10, k=k)),
]:
    docs = fn(test_query, k=TOP_K)
    print("\nMethod:", name)
    print("Top docs:", docs[:5])
    print("nDCG@10:", round(ndcg_at_k(docs, test_qid, 10), 4))
    print("hit_grade3@10:", hit_grade3(docs, test_qid, 10))
    print("semantic_coverage@10:", round(semantic_coverage(docs, test_qid, 10), 4))
    print("sparse_coverage@10:", round(sparse_query_coverage(docs, test_query, 10), 4))

## 21. Define retrieval families and sweep grids

This notebook now treats **fusion, hybrid MMR, hybrid pricing, and dual residual** as parameterized retrieval families.

### Retrieval families
- **dense**: dense embedding similarity only
- **sparse_bm25**: standard BM25 only
- **fusion**: normalized dense and BM25 scores blended by `alpha`
- **hybrid_mmr**: fused relevance minus an MMR diversity penalty with `lambda_div`
- **hybrid_pricing**: fused relevance minus an average-similarity pricing penalty with `lambda_div`
- **dual_residual**: residualized dense + weighted BM25 retrieval with early stopping

### Parameters to sweep
- **fusion**: `alpha`
- **hybrid_mmr**: `alpha`, `lambda_div`
- **hybrid_pricing**: `alpha`, `lambda_div`
- **dual_residual**: `alpha`, `eta`, `sparse_decay`, `beta_stop`, `min_combined_residual_frac`, `min_combined_score`

The grids below are intentionally easy to edit before running the full experiment.


In [ ]:
# =========================
# Parameter grids
# =========================

FUSION_ALPHAS = [0.2, 0.4, 0.6, 0.75, 0.82, 0.90, 0.95]

HYBRID_MMR_ALPHAS = [0.50, 0.75, 0.90]
HYBRID_MMR_LAMBDAS = [0.10, 0.30, 0.50, 0.7]

HYBRID_PRICING_ALPHAS = [0.50, 0.75, 0.90]
HYBRID_PRICING_LAMBDAS = [0.10, 0.30, 0.50, 0.7]

DUALRES_ALPHAS = [0.3, 0.50, 0.75, 0.90]
DUALRES_ETAS = [0.05, 0.10, 0.20, 0.3, 0.4, 0.50]
DUALRES_SPARSE_DECAYS = [0.50, 1.00, 1.6, 2.00]
DUALRES_BETA_STOPS = [0.25, 0.50, 0.75]
DUALRES_MIN_RESIDUALS = [0.05, 0.10, 0.15]
DUALRES_MIN_SCORES = [0.05, 0.10, 0.15]

print("Fusion configs:", len(FUSION_ALPHAS))
print("Hybrid MMR configs:", len(HYBRID_MMR_ALPHAS) * len(HYBRID_MMR_LAMBDAS))
print("Hybrid Pricing configs:", len(HYBRID_PRICING_ALPHAS) * len(HYBRID_PRICING_LAMBDAS))
print("Dual residual configs:", len(DUALRES_ALPHAS) * len(DUALRES_ETAS) * len(DUALRES_SPARSE_DECAYS) * len(DUALRES_BETA_STOPS) * len(DUALRES_MIN_RESIDUALS) * len(DUALRES_MIN_SCORES))


## 22. Method naming helpers

The helper functions below create readable method names and preserve parameter metadata so later analysis can group by family or compare settings directly.


In [ ]:
def fmt_float(x: Optional[float]) -> str:
    if x is None:
        return "none"
    return str(x).replace(".", "")

def fusion_method_name(alpha: float) -> str:
    return f"fusion_a{fmt_float(alpha)}"

def mmr_method_name(alpha: float, lambda_div: float) -> str:
    return f"mmr_a{fmt_float(alpha)}_l{fmt_float(lambda_div)}"

def pricing_method_name(alpha: float, lambda_div: float) -> str:
    return f"pricing_a{fmt_float(alpha)}_l{fmt_float(lambda_div)}"

def dualres_method_name(
    alpha: float,
    eta: float,
    sparse_decay: float,
    beta_stop: float,
    min_combined_residual_frac: float,
    min_combined_score: Optional[float],
) -> str:
    return (
        f"dualres_a{fmt_float(alpha)}"
        f"_eta{fmt_float(eta)}"
        f"_sd{fmt_float(sparse_decay)}"
        f"_b{fmt_float(beta_stop)}"
        f"_r{fmt_float(min_combined_residual_frac)}"
        f"_s{fmt_float(min_combined_score)}"
    )


## 23. Build a unified method registry

Each registry entry stores:
- `method`: the unique config name
- `family`: retrieval family
- `params`: parameter dictionary
- `retriever`: callable used by the evaluation code

This lets the rest of the notebook run one common evaluation pipeline for every method family and every parameter configuration.


In [ ]:
def build_method_registry(top_k: int = TOP_K) -> List[Dict]:
    methods = []

    # Baselines
    methods.append({
        "method": "dense",
        "family": "dense",
        "params": {},
        "retriever": lambda query_text, k=top_k: retrieve_dense(query_text, k=k),
    })
    methods.append({
        "method": "sparse_bm25",
        "family": "sparse_bm25",
        "params": {},
        "retriever": lambda query_text, k=top_k: retrieve_sparse(query_text, k=k),
    })

    # Fusion
    for alpha in FUSION_ALPHAS:
        methods.append({
            "method": fusion_method_name(alpha),
            "family": "fusion",
            "params": {"alpha": alpha},
            "retriever": lambda query_text, alpha=alpha, k=top_k: retrieve_fusion(
                query_text, alpha=alpha, k=k
            ),
        })

    # Hybrid MMR
    for alpha, lambda_div in product(HYBRID_MMR_ALPHAS, HYBRID_MMR_LAMBDAS):
        methods.append({
            "method": mmr_method_name(alpha, lambda_div),
            "family": "hybrid_mmr",
            "params": {"alpha": alpha, "lambda_div": lambda_div},
            "retriever": lambda query_text, alpha=alpha, lambda_div=lambda_div, k=top_k: retrieve_hybrid_mmr(
                query_text, alpha=alpha, lambda_div=lambda_div, k=k
            ),
        })

    # Hybrid Pricing
    for alpha, lambda_div in product(HYBRID_PRICING_ALPHAS, HYBRID_PRICING_LAMBDAS):
        methods.append({
            "method": pricing_method_name(alpha, lambda_div),
            "family": "hybrid_pricing",
            "params": {"alpha": alpha, "lambda_div": lambda_div},
            "retriever": lambda query_text, alpha=alpha, lambda_div=lambda_div, k=top_k: retrieve_hybrid_pricing(
                query_text, alpha=alpha, lambda_div=lambda_div, k=k
            ),
        })

    # Dual Residual
    for alpha, eta, sparse_decay, beta_stop, min_resid, min_score in product(
        DUALRES_ALPHAS,
        DUALRES_ETAS,
        DUALRES_SPARSE_DECAYS,
        DUALRES_BETA_STOPS,
        DUALRES_MIN_RESIDUALS,
        DUALRES_MIN_SCORES,
    ):
        methods.append({
            "method": dualres_method_name(
                alpha=alpha,
                eta=eta,
                sparse_decay=sparse_decay,
                beta_stop=beta_stop,
                min_combined_residual_frac=min_resid,
                min_combined_score=min_score,
            ),
            "family": "dual_residual",
            "params": {
                "alpha": alpha,
                "eta": eta,
                "sparse_decay": sparse_decay,
                "beta_stop": beta_stop,
                "min_combined_residual_frac": min_resid,
                "min_combined_score": min_score,
            },
            "retriever": lambda query_text,
                               alpha=alpha,
                               eta=eta,
                               sparse_decay=sparse_decay,
                               beta_stop=beta_stop,
                               min_resid=min_resid,
                               min_score=min_score,
                               k=top_k: retrieve_dual_residual(
                                    query_text,
                                    alpha=alpha,
                                    eta=eta,
                                    sparse_decay=sparse_decay,
                                    beta_stop=beta_stop,
                                    min_combined_residual_frac=min_resid,
                                    min_combined_score=min_score,
                                    k=k,
                               ),
        })

    return methods

method_registry = build_method_registry(top_k=TOP_K)
len(method_registry)


## 24. Inspect a few registered methods


In [ ]:
method_registry = build_method_registry(top_k=TOP_K)

print("Registered method configs:", len(method_registry))
pd.DataFrame([
    {
        "method": entry["method"],
        "family": entry["family"],
        **entry.get("params", {})
    }
    for entry in method_registry
]).head(10)

In [ ]:
for entry in method_registry[:10]:
    print(entry["method"], entry["family"], entry["params"])


## 25. Unified benchmark runner with checkpointing

The next functions support both ordinary runs and restartable large sweeps.

The checkpointed runner:
1. saves per-method progress to disk while the run is happening,
2. can resume a partially completed method by skipping already-finished query IDs,
3. rebuilds the long-form results table from checkpoint files,
4. writes refreshed aggregate CSVs after each method so a preempted job can be resumed cleanly.


In [ ]:
def run_method_registry(
    method_registry: List[Dict],
    qids: Optional[List[str]] = None,
    k: int = TOP_K,
    verbose: bool = True,
) -> pd.DataFrame:
    all_rows = []

    for entry in method_registry:
        method_name = entry["method"]
        family = entry["family"]
        params = entry["params"]
        retriever_fn = entry["retriever"]

        df = evaluate_method_over_queries(
            method_name=method_name,
            retrieve_fn=retriever_fn,
            qids=qids,
            k=k,
            verbose=verbose,
        )

        df["family"] = family
        for key, value in params.items():
            df[key] = value

        all_rows.append(df)

    return pd.concat(all_rows, ignore_index=True)

def collect_checkpoint_results(
    method_registry: Optional[List[Dict]] = None,
    checkpoint_dir: str = CHECKPOINT_DIR,
) -> pd.DataFrame:
    if method_registry is None:
        method_registry = []

    discovered_paths = sorted(glob.glob(os.path.join(checkpoint_dir, "*.csv")))
    skip_names = {
        os.path.basename(RESULTS_ALL_METHODS_CSV),
        os.path.basename(SUMMARY_ALL_METHODS_CSV),
        os.path.basename(BEST_FAMILY_CSV),
    }
    discovered_paths = [p for p in discovered_paths if os.path.basename(p) not in skip_names]

    dfs = []
    for path in discovered_paths:
        df = pd.read_csv(path)
        dfs.append(df)

    if not dfs:
        return pd.DataFrame()

    results_df = pd.concat(dfs, ignore_index=True)

    # Keep method ordering aligned with the current registry when possible
    if method_registry and "method" in results_df.columns:
        method_order = {entry["method"]: i for i, entry in enumerate(method_registry)}
        results_df["__method_order"] = results_df["method"].map(method_order).fillna(10**9)
        results_df = results_df.sort_values(["__method_order", "method", "qid"]).drop(columns="__method_order")

    return results_df.reset_index(drop=True)

def write_checkpoint_manifest(
    method_registry: List[Dict],
    qids: Optional[List[str]],
    k: int,
    checkpoint_dir: str = CHECKPOINT_DIR,
) -> None:
    manifest = {
        "num_methods": len(method_registry),
        "method_names": [entry["method"] for entry in method_registry],
        "num_qids": None if qids is None else len(qids),
        "qids": None if qids is None else [str(qid) for qid in qids],
        "k": k,
    }
    with open(CHECKPOINT_MANIFEST, "w") as f:
        json.dump(manifest, f, indent=2)

def run_method_registry_checkpointed(
    method_registry: List[Dict],
    qids: Optional[List[str]] = None,
    k: int = TOP_K,
    verbose: bool = True,
    checkpoint_dir: str = CHECKPOINT_DIR,
    flush_every: int = 1,
    aggregate_results_csv: str = RESULTS_ALL_METHODS_CSV,
    force_restart_method: bool = False,
) -> pd.DataFrame:
    os.makedirs(checkpoint_dir, exist_ok=True)
    write_checkpoint_manifest(method_registry=method_registry, qids=qids, k=k, checkpoint_dir=checkpoint_dir)

    aggregate_dirty = False

    for entry in method_registry:
        method_name = entry["method"]
        family = entry["family"]
        params = entry["params"]
        retriever_fn = entry["retriever"]

        print(f"Running method: {method_name}")

        method_df, wrote_anything = evaluate_method_over_queries_checkpointed(
            method_name=method_name,
            retrieve_fn=retriever_fn,
            qids=qids,
            k=k,
            verbose=verbose,
            family=family,
            params=params,
            checkpoint_dir=checkpoint_dir,
            flush_every=flush_every,
            force_restart_method=force_restart_method,
        )

        if wrote_anything:
            aggregate_dirty = True
            checkpoint_results_df = collect_checkpoint_results(
                method_registry=method_registry,
                checkpoint_dir=checkpoint_dir,
            )
            atomic_to_csv(checkpoint_results_df, aggregate_results_csv)
            print(f"Checkpointed aggregate results to: {aggregate_results_csv}")
        else:
            print(f"No new rows for {method_name}; skipped aggregate rebuild.")

    if aggregate_dirty or (not os.path.exists(aggregate_results_csv)):
        final_df = collect_checkpoint_results(
            method_registry=method_registry,
            checkpoint_dir=checkpoint_dir,
        )
        atomic_to_csv(final_df, aggregate_results_csv)
        return final_df

    return pd.read_csv(aggregate_results_csv)

def mean_ignore_none(series: pd.Series) -> float:
    vals = pd.to_numeric(series, errors="coerce")
    vals = vals.dropna()
    return float(vals.mean()) if len(vals) else np.nan

def summarize_results(results_df: pd.DataFrame) -> pd.DataFrame:
    metric_cols = [
        "nDCG@k",
        "irrelevant@k",
        "redundancy@k",
        "hit_grade3@k",
        "semantic_coverage@k",
        "sparse_coverage@k",
        "docs_retrieved",
    ]

    summary = (
        results_df
        .groupby(["method", "family"], as_index=False)[metric_cols]
        .mean()
    )

    for thr in SEMANTIC_THRESHOLDS:
        col = f"docs_to_semantic_{thr:.2f}"
        extra = (
            results_df.groupby(["method", "family"])[col]
            .apply(mean_ignore_none)
            .reset_index(name=col)
        )
        summary = summary.merge(extra, on=["method", "family"], how="left")

    for thr in SPARSE_THRESHOLDS:
        col = f"docs_to_sparse_{thr:.2f}"
        extra = (
            results_df.groupby(["method", "family"])[col]
            .apply(mean_ignore_none)
            .reset_index(name=col)
        )
        summary = summary.merge(extra, on=["method", "family"], how="left")

    param_cols = [
        "alpha",
        "lambda_div",
        "eta",
        "sparse_decay",
        "beta_stop",
        "min_combined_residual_frac",
        "min_combined_score",
    ]
    available_param_cols = [c for c in param_cols if c in results_df.columns]
    if available_param_cols:
        params_df = (
            results_df[["method", "family"] + available_param_cols]
            .drop_duplicates(subset=["method", "family"])
        )
        summary = summary.merge(params_df, on=["method", "family"], how="left")

    return summary

def clear_checkpoint_dir(checkpoint_dir: str = CHECKPOINT_DIR) -> None:
    if os.path.exists(checkpoint_dir):
        for path in glob.glob(os.path.join(checkpoint_dir, "*.csv")):
            os.remove(path)
        if os.path.exists(CHECKPOINT_MANIFEST):
            os.remove(CHECKPOINT_MANIFEST)
    print("Cleared checkpoint files in:", checkpoint_dir)

## 26. Choose a run mode

A full sweep can be large. Start with a pilot run on a subset of queries, then run the full experiment once the code and metrics look correct.

For long Marlowe runs, prefer the checkpointed runner so that if the job is preempted you can restart and continue from the saved per-method and per-query checkpoint files.


In [ ]:
PILOT_NUM_QUERIES = 20
pilot_qids = list(queries_frozen.keys())[:PILOT_NUM_QUERIES]
full_qids = list(queries_frozen.keys())

print("Pilot queries:", len(pilot_qids))
print("Full queries:", len(full_qids))
print("Total method configs:", len(method_registry))


## 27. Pilot sweep

This runs all retrieval families and all configured parameter sweeps on a smaller set of queries first.

Two options are shown below:
- a standard in-memory pilot run,
- a checkpointed pilot run that is restartable.


In [ ]:
# # Standard pilot run
# pilot_results_df = run_method_registry(method_registry, qids=pilot_qids, k=TOP_K, verbose=True)
# pilot_results_path = os.path.join(CACHE_DIR, "pilot_results_all_methods.csv")
# pilot_results_df.to_csv(pilot_results_path, index=False)
# pilot_summary_df = summarize_results(pilot_results_df)
# pilot_summary_df.sort_values(["family", "nDCG@k"], ascending=[True, False]).head(30)

# # Restartable pilot run
# pilot_checkpoint_dir = os.path.join(CACHE_DIR, "pilot_checkpoints")
# os.makedirs(pilot_checkpoint_dir, exist_ok=True)
# pilot_results_df = run_method_registry_checkpointed(
#     method_registry=method_registry,
#     qids=pilot_qids,
#     k=TOP_K,
#     verbose=True,
#     checkpoint_dir=pilot_checkpoint_dir,
#     flush_every=1,
#     aggregate_results_csv=os.path.join(CACHE_DIR, "pilot_results_all_methods_checkpointed.csv"),
# )
# pilot_summary_df = summarize_results(pilot_results_df)
# pilot_summary_df.sort_values(["family", "nDCG@k"], ascending=[True, False]).head(30)

In [ ]:
import os
import re
import glob
import json
from typing import Dict, List, Optional

import pandas as pd
import numpy as np

CACHE_DIR = "rag_diversity_cache"
CHECKPOINT_DIR = os.path.join(CACHE_DIR, "checkpoints")

RESULTS_CSV = os.path.join(CACHE_DIR, "method_results.csv")
RESULTS_ALL_METHODS_CSV = os.path.join(CACHE_DIR, "all_methods_results.csv")
SUMMARY_ALL_METHODS_CSV = os.path.join(CACHE_DIR, "all_methods_summary.csv")
BEST_FAMILY_CSV = os.path.join(CACHE_DIR, "best_per_family.csv")
CHECKPOINT_MANIFEST = os.path.join(CHECKPOINT_DIR, "checkpoint_manifest.json")

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("CACHE_DIR =", CACHE_DIR)
print("CHECKPOINT_DIR =", CHECKPOINT_DIR)
print("RESULTS_ALL_METHODS_CSV =", RESULTS_ALL_METHODS_CSV)
print("SUMMARY_ALL_METHODS_CSV =", SUMMARY_ALL_METHODS_CSV)
print("BEST_FAMILY_CSV =", BEST_FAMILY_CSV)
print("CHECKPOINT_MANIFEST =", CHECKPOINT_MANIFEST)

In [ ]:
import re

def safe_method_filename(name):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", name)

preview = []

for entry in method_registry:
    method_name = entry["method"]
    preview.append({
        "method": method_name,
        "safe_name": safe_method_filename(method_name)
    })

preview_df = pd.DataFrame(preview)

print("duplicate filenames:", preview_df["safe_name"].duplicated().any())
preview_df.head(10)

In [ ]:
# Option A: load the aggregate long-form CSV if it exists
# results_df = pd.read_csv(RESULTS_ALL_METHODS_CSV)
# results_df.head()

# Option B: rebuild the long-form table directly from per-method checkpoint CSVs
results_df = collect_checkpoint_results(method_registry=method_registry, checkpoint_dir=CHECKPOINT_DIR)
atomic_to_csv(results_df, RESULTS_ALL_METHODS_CSV)
results_df

In [ ]:
import pandas as pd

df = pd.read_csv(method_checkpoint_path("dense"))
print("rows:", len(df))
print("unique qids:", df["qid"].nunique())
print("duplicate qids:", len(df) - df["qid"].nunique())

In [ ]:
import glob
import os
import pandas as pd

rows = []
for path in glob.glob(os.path.join(CHECKPOINT_DIR, "*.csv")):
    df = pd.read_csv(path)
    rows.append({
        "file": os.path.basename(path),
        "rows": len(df),
        "unique_qids": df["qid"].nunique() if "qid" in df.columns else None,
        "duplicate_qids": len(df) - df["qid"].nunique() if "qid" in df.columns else None,
    })

pd.DataFrame(rows).sort_values("duplicate_qids", ascending=False).head(20)

In [ ]:
results_df = run_method_registry_checkpointed(
    method_registry=method_registry,
    qids=full_qids,
    k=TOP_K,
    verbose=True,
    checkpoint_dir=CHECKPOINT_DIR,
    flush_every=1,
    aggregate_results_csv=RESULTS_ALL_METHODS_CSV,
)

print("Checkpointed full results to:", RESULTS_ALL_METHODS_CSV)
results_df.head()

In [ ]:
metric_cols = [
    "nDCG@k",
    "irrelevant@k",
    "redundancy@k",
    "hit_grade3@k",
    "semantic_coverage@k",
    "sparse_coverage@k",
    "docs_retrieved",
    "docs_to_semantic_0.80",
    "docs_to_semantic_0.90",
    "docs_to_semantic_0.95",
    "docs_to_sparse_0.80",
    "docs_to_sparse_0.90",
    "docs_to_sparse_0.95",
]

param_cols = [
    "alpha",
    "lambda_div",
    "eta",
    "sparse_decay",
    "beta_stop",
    "min_combined_residual_frac",
    "min_combined_score",
]

available_metric_cols = [c for c in metric_cols if c in results_df.columns]
available_param_cols = [c for c in param_cols if c in results_df.columns]

summary_df = (
    results_df
    .groupby(["method", "family"], as_index=False)[available_metric_cols]
    .mean()
)

if available_param_cols:
    params_df = (
        results_df[["method", "family"] + available_param_cols]
        .drop_duplicates(subset=["method", "family"])
    )
    summary_df = summary_df.merge(params_df, on=["method", "family"], how="left")

In [ ]:
summary_df

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make a working copy
analysis_df = summary_df.copy()

# Standardize family names just in case
analysis_df["family"] = analysis_df["family"].astype(str)

# Numeric columns we care about
metric_cols = [
    "nDCG@k",
    "irrelevant@k",
    "redundancy@k",
    "hit_grade3@k",
    "semantic_coverage@k",
    "sparse_coverage@k",
    "docs_retrieved",
    "docs_to_semantic_0.80",
    "docs_to_semantic_0.90",
    "docs_to_semantic_0.95",
    "docs_to_sparse_0.80",
    "docs_to_sparse_0.90",
    "docs_to_sparse_0.95",
]

param_cols = [
    "alpha",
    "lambda_div",
    "eta",
    "sparse_decay",
    "beta_stop",
    "min_combined_residual_frac",
    "min_combined_score",
]

for col in metric_cols + param_cols:
    if col in analysis_df.columns:
        analysis_df[col] = pd.to_numeric(analysis_df[col], errors="coerce")

print("Families:", sorted(analysis_df["family"].dropna().unique()))
print("Rows:", len(analysis_df))
analysis_df.head()

In [ ]:
def best_per_family(
    df: pd.DataFrame,
    primary: str = "nDCG@k",
    ascending_primary: bool = False,
    secondary: list = None,
):
    """
    Return one best row per family.
    secondary should be a list of tuples: (col, ascending)
    """
    if secondary is None:
        secondary = [("redundancy@k", True)]

    sort_cols = ["family", primary] + [c for c, _ in secondary]
    ascending = [True, ascending_primary] + [a for _, a in secondary]

    best_df = (
        df.sort_values(sort_cols, ascending=ascending)
          .groupby("family", as_index=False)
          .head(1)
          .reset_index(drop=True)
    )
    return best_df


def show_best_table(df: pd.DataFrame):
    keep_cols = [
        "family",
        "method",
        "nDCG@k",
        "redundancy@k",
        "hit_grade3@k",
        "semantic_coverage@k",
        "sparse_coverage@k",
        "irrelevant@k",
        "docs_retrieved",
        "docs_to_semantic_0.80",
        "docs_to_semantic_0.90",
        "docs_to_sparse_0.80",
        "docs_to_sparse_0.90",
        "alpha",
        "lambda_div",
        "eta",
        "sparse_decay",
        "beta_stop",
        "min_combined_residual_frac",
        "min_combined_score",
    ]
    keep_cols = [c for c in keep_cols if c in df.columns]
    return df[keep_cols].sort_values("family").reset_index(drop=True)

In [ ]:
best_by_ndcg = best_per_family(
    analysis_df,
    primary="nDCG@k",
    ascending_primary=False,
    secondary=[
        ("redundancy@k", True),
        ("irrelevant@k", True),
    ],
)

best_by_ndcg_table = show_best_table(best_by_ndcg)
best_by_ndcg_table

In [ ]:
analysis_df["balanced_score"] = (
    0.40 * analysis_df["nDCG@k"].fillna(0)
    + 0.20 * analysis_df["hit_grade3@k"].fillna(0)
    + 0.20 * analysis_df["semantic_coverage@k"].fillna(0)
    + 0.20 * analysis_df["sparse_coverage@k"].fillna(0)
    - 0.20 * analysis_df["redundancy@k"].fillna(0)
    - 0.10 * analysis_df["irrelevant@k"].fillna(0) / max(1, analysis_df["irrelevant@k"].max())
)

best_by_balanced = best_per_family(
    analysis_df,
    primary="balanced_score",
    ascending_primary=False,
    secondary=[
        ("nDCG@k", False),
        ("redundancy@k", True),
    ],
)

best_by_balanced_table = show_best_table(best_by_balanced)
best_by_balanced_table

In [ ]:
analysis_df["coverage_efficiency_score"] = (
    0.35 * analysis_df["semantic_coverage@k"].fillna(0)
    + 0.35 * analysis_df["sparse_coverage@k"].fillna(0)
    + 0.15 * analysis_df["hit_grade3@k"].fillna(0)
    - 0.15 * analysis_df["docs_retrieved"].fillna(analysis_df["docs_retrieved"].max()) / max(1, analysis_df["docs_retrieved"].max())
    - 0.15 * analysis_df["redundancy@k"].fillna(0)
)

best_by_coverage = best_per_family(
    analysis_df,
    primary="coverage_efficiency_score",
    ascending_primary=False,
    secondary=[
        ("semantic_coverage@k", False),
        ("sparse_coverage@k", False),
        ("redundancy@k", True),
    ],
)

best_by_coverage_table = show_best_table(best_by_coverage)
best_by_coverage_table

In [ ]:
best_family_table = best_by_balanced_table.copy()
best_family_table

In [ ]:
def scatter_metric_tradeoff(
    df: pd.DataFrame,
    x: str,
    y: str,
    title: str,
    annotate_best_per_family: bool = True,
    best_df: pd.DataFrame = None,
):
    plt.figure(figsize=(10, 7))

    families = sorted(df["family"].dropna().unique())
    for fam in families:
        sub = df[df["family"] == fam]
        plt.scatter(sub[x], sub[y], label=fam, alpha=0.7)

    if annotate_best_per_family and best_df is not None:
        for _, row in best_df.iterrows():
            if pd.notna(row[x]) and pd.notna(row[y]):
                plt.annotate(
                    row["family"],
                    (row[x], row[y]),
                    xytext=(5, 5),
                    textcoords="offset points",
                    fontsize=9
                )

    plt.xlabel(x)
    plt.ylabel(y)
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


scatter_metric_tradeoff(
    analysis_df,
    x="redundancy@k",
    y="nDCG@k",
    title="Redundancy vs nDCG",
    annotate_best_per_family=True,
    best_df=best_by_balanced,
)

In [ ]:
scatter_metric_tradeoff(
    analysis_df,
    x="redundancy@k",
    y="semantic_coverage@k",
    title="Redundancy vs Semantic Coverage",
    annotate_best_per_family=True,
    best_df=best_by_balanced,
)

In [ ]:
scatter_metric_tradeoff(
    analysis_df,
    x="redundancy@k",
    y="hit_grade3@k",
    title="Redundancy vs Grade-3 Hit Rate",
    annotate_best_per_family=True,
    best_df=best_by_balanced,
)

In [ ]:
scatter_metric_tradeoff(
    analysis_df,
    x="docs_retrieved",
    y="semantic_coverage@k",
    title="Docs Retrieved vs Semantic Coverage",
    annotate_best_per_family=True,
    best_df=best_by_balanced,
)

scatter_metric_tradeoff(
    analysis_df,
    x="docs_retrieved",
    y="sparse_coverage@k",
    title="Docs Retrieved vs Sparse Coverage",
    annotate_best_per_family=True,
    best_df=best_by_balanced,
)

In [ ]:
family_summary = (
    analysis_df
    .groupby("family", as_index=False)[[
        "nDCG@k",
        "irrelevant@k",
        "redundancy@k",
        "hit_grade3@k",
        "semantic_coverage@k",
        "sparse_coverage@k",
        "docs_retrieved",
    ]]
    .agg(["mean", "max", "min"])
)

family_summary

In [ ]:
def plot_best_family_bars(best_df: pd.DataFrame, metric: str, title: str):
    plot_df = best_df.sort_values(metric, ascending=False)

    plt.figure(figsize=(10, 6))
    plt.bar(plot_df["family"], plot_df[metric])
    plt.xticks(rotation=30, ha="right")
    plt.ylabel(metric)
    plt.title(title)
    plt.grid(axis="y", alpha=0.3)
    plt.show()


plot_best_family_bars(best_by_balanced, "nDCG@k", "Best Config per Family: nDCG")
plot_best_family_bars(best_by_balanced, "redundancy@k", "Best Config per Family: Redundancy")
plot_best_family_bars(best_by_balanced, "semantic_coverage@k", "Best Config per Family: Semantic Coverage")
plot_best_family_bars(best_by_balanced, "sparse_coverage@k", "Best Config per Family: Sparse Coverage")
plot_best_family_bars(best_by_balanced, "hit_grade3@k", "Best Config per Family: Grade-3 Hit Rate")

In [ ]:
def pareto_frontier(df: pd.DataFrame, x: str, y: str, minimize_x: bool = True, maximize_y: bool = True):
    points = df[[x, y]].dropna().copy()
    idxs = points.index.tolist()

    frontier = []
    for i in idxs:
        xi, yi = df.loc[i, x], df.loc[i, y]
        dominated = False
        for j in idxs:
            if i == j:
                continue
            xj, yj = df.loc[j, x], df.loc[j, y]

            x_better_or_equal = xj <= xi if minimize_x else xj >= xi
            y_better_or_equal = yj >= yi if maximize_y else yj <= yi

            x_strict = xj < xi if minimize_x else xj > xi
            y_strict = yj > yi if maximize_y else yj < yi

            if x_better_or_equal and y_better_or_equal and (x_strict or y_strict):
                dominated = True
                break
        if not dominated:
            frontier.append(i)

    return df.loc[frontier].copy()


pareto_ndcg_red = pareto_frontier(
    analysis_df,
    x="redundancy@k",
    y="nDCG@k",
    minimize_x=True,
    maximize_y=True,
).sort_values("redundancy@k")

pareto_ndcg_red[["family", "method", "nDCG@k", "redundancy@k", "hit_grade3@k", "semantic_coverage@k"]]

In [ ]:
plt.figure(figsize=(10, 7))

for fam in sorted(analysis_df["family"].dropna().unique()):
    sub = analysis_df[analysis_df["family"] == fam]
    plt.scatter(sub["redundancy@k"], sub["nDCG@k"], alpha=0.4, label=fam)

plt.plot(
    pareto_ndcg_red["redundancy@k"],
    pareto_ndcg_red["nDCG@k"],
    marker="o"
)

for _, row in pareto_ndcg_red.iterrows():
    plt.annotate(
        row["family"],
        (row["redundancy@k"], row["nDCG@k"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8
    )

plt.xlabel("redundancy@k")
plt.ylabel("nDCG@k")
plt.title("Pareto Frontier: Redundancy vs nDCG")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
def top_configs_per_family(df: pd.DataFrame, metric: str = "nDCG@k", top_n: int = 5):
    out = (
        df.sort_values(["family", metric], ascending=[True, False])
          .groupby("family", as_index=False)
          .head(top_n)
          .reset_index(drop=True)
    )
    keep_cols = [
        "family",
        "method",
        metric,
        "redundancy@k",
        "hit_grade3@k",
        "semantic_coverage@k",
        "sparse_coverage@k",
        "irrelevant@k",
        "docs_retrieved",
        "alpha",
        "lambda_div",
        "eta",
        "sparse_decay",
        "beta_stop",
        "min_combined_residual_frac",
        "min_combined_score",
    ]
    keep_cols = [c for c in keep_cols if c in out.columns]
    return out[keep_cols]


top_ndcg_per_family = top_configs_per_family(analysis_df, metric="nDCG@k", top_n=5)
top_ndcg_per_family

In [ ]:
top_balanced_per_family = (
    analysis_df.sort_values(["family", "balanced_score"], ascending=[True, False])
               .groupby("family", as_index=False)
               .head(5)
               .reset_index(drop=True)
)

show_best_table(top_balanced_per_family)

In [ ]:
dual_df = analysis_df[analysis_df["family"] == "dual_residual"].copy()

dual_df.sort_values("nDCG@k", ascending=False).head(10)[[
    "method",
    "nDCG@k",
    "redundancy@k",
    "hit_grade3@k",
    "semantic_coverage@k",
    "sparse_coverage@k",
    "docs_retrieved",
    "alpha",
    "eta",
    "sparse_decay",
    "beta_stop",
    "min_combined_residual_frac",
    "min_combined_score",
]]

In [ ]:
if len(dual_df):
    dual_eta_summary = dual_df.groupby("eta", as_index=False)[[
        "nDCG@k",
        "redundancy@k",
        "hit_grade3@k",
        "semantic_coverage@k",
        "sparse_coverage@k",
        "docs_retrieved",
    ]].mean()
    display(dual_eta_summary)

    dual_alpha_summary = dual_df.groupby("alpha", as_index=False)[[
        "nDCG@k",
        "redundancy@k",
        "hit_grade3@k",
        "semantic_coverage@k",
        "sparse_coverage@k",
        "docs_retrieved",
    ]].mean()
    display(dual_alpha_summary)

In [ ]:
if len(dual_df):
    plt.figure(figsize=(8, 5))
    eta_means = dual_df.groupby("eta")["nDCG@k"].mean().sort_index()
    plt.plot(eta_means.index, eta_means.values, marker="o")
    plt.xlabel("eta")
    plt.ylabel("mean nDCG@k")
    plt.title("Dual Residual: eta vs nDCG")
    plt.grid(True, alpha=0.3)
    plt.show()

    plt.figure(figsize=(8, 5))
    eta_red = dual_df.groupby("eta")["redundancy@k"].mean().sort_index()
    plt.plot(eta_red.index, eta_red.values, marker="o")
    plt.xlabel("eta")
    plt.ylabel("mean redundancy@k")
    plt.title("Dual Residual: eta vs Redundancy")
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
for fam in ["hybrid_mmr", "hybrid_pricing", "fusion"]:
    sub = analysis_df[analysis_df["family"] == fam].copy()
    if len(sub) == 0:
        continue

    print(f"\n===== {fam} =====")
    display(
        sub.sort_values("nDCG@k", ascending=False).head(10)[[
            c for c in [
                "method",
                "nDCG@k",
                "redundancy@k",
                "hit_grade3@k",
                "semantic_coverage@k",
                "sparse_coverage@k",
                "irrelevant@k",
                "alpha",
                "lambda_div",
            ] if c in sub.columns
        ]]
    )

In [ ]:
for fam in ["hybrid_mmr", "hybrid_pricing"]:
    sub = analysis_df[analysis_df["family"] == fam].copy()
    if len(sub) == 0 or "lambda_div" not in sub.columns:
        continue

    print(f"\n===== {fam}: mean metrics by lambda_div =====")
    display(
        sub.groupby("lambda_div", as_index=False)[[
            "nDCG@k",
            "redundancy@k",
            "hit_grade3@k",
            "semantic_coverage@k",
            "sparse_coverage@k",
            "irrelevant@k",
        ]].mean()
    )

In [ ]:
paper_best_table = best_by_balanced[[
    c for c in [
        "family",
        "method",
        "nDCG@k",
        "irrelevant@k",
        "redundancy@k",
        "hit_grade3@k",
        "semantic_coverage@k",
        "sparse_coverage@k",
        "docs_retrieved",
        "docs_to_semantic_0.80",
        "docs_to_semantic_0.90",
        "docs_to_sparse_0.80",
        "docs_to_sparse_0.90",
    ] if c in best_by_balanced.columns
]].sort_values("family").reset_index(drop=True)

paper_best_table

## 29. Reload an existing run or rebuild from checkpoints

Use the cells below if:
- you already completed the sweep,
- the job was preempted and you want to inspect whatever was saved so far,
- or you restarted the notebook and want to continue from checkpoint files.


In [ ]:
# Option A: load the aggregate long-form CSV if it exists
# results_df = pd.read_csv(RESULTS_ALL_METHODS_CSV)
# results_df.head()

# Option B: rebuild the long-form table directly from per-method checkpoint CSVs
# results_df = collect_checkpoint_results(method_registry=method_registry, checkpoint_dir=CHECKPOINT_DIR)
# atomic_to_csv(results_df, RESULTS_ALL_METHODS_CSV)
# results_df.head()

## 30. Aggregate method-level results

This produces one row per method configuration with all metrics:
- ranking quality
- irrelevance
- redundancy
- grade-3 hit rate
- semantic coverage
- sparse coverage
- efficiency thresholds


In [ ]:
# summary_df = summarize_results(results_df)
# summary_df = summary_df.sort_values("nDCG@k", ascending=False).reset_index(drop=True)
# summary_df.round(4).head(25)


## 31. Best configuration per family

This is often the most useful paper-facing summary. You can choose the criterion depending on the story you want to tell.

Below, the default criterion is:
1. highest `nDCG@k`
2. then highest `semantic_coverage@k`
3. then lowest `redundancy@k`


In [ ]:
def best_config_per_family(
    summary_df: pd.DataFrame,
    sort_cols: Optional[List[str]] = None,
    ascending: Optional[List[bool]] = None,
) -> pd.DataFrame:
    if sort_cols is None:
        sort_cols = ["nDCG@k", "semantic_coverage@k", "redundancy@k"]
    if ascending is None:
        ascending = [False, False, True]

    pieces = []
    for family, group in summary_df.groupby("family"):
        best = group.sort_values(sort_cols, ascending=ascending).head(1)
        pieces.append(best)
    return pd.concat(pieces, ignore_index=True)

# best_family_df = best_config_per_family(summary_df)
# best_family_df.round(4)


## 32. Alternative best-config views

Sometimes you want the best configuration per family under a different objective, such as:
- highest semantic coverage
- highest grade-3 hit rate
- best low-redundancy coverage tradeoff


In [ ]:
def best_by_metric(summary_df: pd.DataFrame, metric: str, higher_is_better: bool = True) -> pd.DataFrame:
    ascending = not higher_is_better
    pieces = []
    for family, group in summary_df.groupby("family"):
        best = group.sort_values(metric, ascending=ascending).head(1)
        pieces.append(best)
    return pd.concat(pieces, ignore_index=True)

# best_semantic_df = best_by_metric(summary_df, "semantic_coverage@k", higher_is_better=True)
# best_hit3_df = best_by_metric(summary_df, "hit_grade3@k", higher_is_better=True)
# best_low_redundancy_df = best_by_metric(summary_df, "redundancy@k", higher_is_better=False)


## 33. Family-level sweep summaries

These compact summaries are helpful for seeing what the parameter sweeps are doing before inspecting individual configurations.


In [ ]:
def family_summary_tables(results_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    out = {}

    if "alpha" in results_df.columns:
        out["alpha_by_family"] = (
            results_df.dropna(subset=["alpha"])
            .groupby(["family", "alpha"])[["nDCG@k", "redundancy@k", "semantic_coverage@k", "hit_grade3@k", "sparse_coverage@k"]]
            .mean()
            .reset_index()
            .sort_values(["family", "alpha"])
        )

    if {"family", "lambda_div"}.issubset(results_df.columns):
        out["lambda_div_by_family"] = (
            results_df[results_df["family"].isin(["hybrid_mmr", "hybrid_pricing"])]
            .dropna(subset=["lambda_div"])
            .groupby(["family", "alpha", "lambda_div"])[["nDCG@k", "redundancy@k", "semantic_coverage@k", "hit_grade3@k"]]
            .mean()
            .reset_index()
            .sort_values(["family", "alpha", "lambda_div"])
        )

    if "eta" in results_df.columns:
        out["dualres_eta"] = (
            results_df[results_df["family"] == "dual_residual"]
            .dropna(subset=["eta"])
            .groupby(["eta"])[["nDCG@k", "redundancy@k", "semantic_coverage@k", "hit_grade3@k", "docs_retrieved"]]
            .mean()
            .reset_index()
            .sort_values("eta")
        )

    return out

# family_tables = family_summary_tables(results_df)
# for name, table in family_tables.items():
#     print("\n", name)
#     display(table.head(20))


## 34. Visualize the tradeoffs

Two especially useful plots:
- `nDCG@k` versus `redundancy@k`
- `semantic_coverage@k` versus `docs_retrieved`

The first shows ranking-diversity tradeoffs.  
The second shows coverage-efficiency tradeoffs.


In [ ]:
def plot_tradeoff(
    summary_df: pd.DataFrame,
    x_col: str,
    y_col: str,
    title: str,
    label_points: bool = False,
):
    plt.figure(figsize=(8, 6))
    families = summary_df["family"].unique()

    for family in families:
        sub = summary_df[summary_df["family"] == family]
        plt.scatter(sub[x_col], sub[y_col], label=family, alpha=0.75)
        if label_points:
            for _, row in sub.iterrows():
                plt.annotate(row["method"], (row[x_col], row[y_col]), fontsize=7, alpha=0.7)

    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

# plot_tradeoff(summary_df, "redundancy@k", "nDCG@k", "nDCG vs redundancy across all configs", label_points=False)
# plot_tradeoff(summary_df, "docs_retrieved", "semantic_coverage@k", "Semantic coverage vs docs retrieved", label_points=False)


## 35. Export paper-facing tables

This exports:
- the long-form per-query results table,
- the aggregate summary over every method configuration,
- the best configuration per family.

The paths below are aligned with the checkpointed full-run defaults.


In [ ]:
# export_dir = CACHE_DIR
# long_results_path = RESULTS_ALL_METHODS_CSV
# summary_results_path = SUMMARY_ALL_METHODS_CSV
# best_family_path = BEST_FAMILY_CSV

# results_df.to_csv(long_results_path, index=False)
# summary_df.to_csv(summary_results_path, index=False)
# best_family_df.to_csv(best_family_path, index=False)

# print("Saved long-form results to:", long_results_path)
# print("Saved summary results to:", summary_results_path)
# print("Saved best-per-family table to:", best_family_path)

## 36. Minimal run checklist

1. Run the dataset-loading and candidate-pool cells.
2. Build BM25 and dense embeddings.
3. Run a quick spot-check on one query.
4. Edit the sweep grids if needed.
5. Run the pilot sweep first.
6. Inspect `pilot_summary_df` and the family-level tables.
7. Run the checkpointed full sweep.
8. If the cluster preempts the job, restart the notebook and rerun the same checkpointed full-sweep cell.
9. Rebuild or reload `results_df` from checkpoint files if needed.
10. Export `results_df`, `summary_df`, and `best_family_df`.

This notebook is now structured so the same evaluation pipeline works for:
- clearly defined retrieval methods,
- parameter sweeps for fusion, hybrid MMR, hybrid pricing, and dual residual,
- ranking, redundancy, coverage, and efficiency metrics,
- checkpointed large runs that can resume after interruption.


## Embedding-space spread experiment

This section is appended **after** the original pipeline, so it uses the same:
- dataset download / load
- frozen query set
- candidate document pool
- dense document embeddings
- retrieval functions
- method registry / parameter sweep

Run the notebook **top to bottom**. If the kernel was restarted, use **Restart & Run All** before running this section.

In [ ]:
# Sanity check: make sure the required earlier pipeline objects exist
required_names = [
    "CACHE_DIR",
    "TOP_K",
    "queries_frozen",
    "qrels_frozen",
    "candidate_doc_ids",
    "doc_index",
    "doc_embeddings",
    "method_registry",
    "full_qids",
]

missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(
        "This section depends on earlier cells in the notebook. "
        "Missing globals: " + ", ".join(missing) + ". "
        "Please run the notebook from the top (Kernel -> Restart & Run All)."
    )

print("All required pipeline globals are available.")
print("Queries:", len(queries_frozen))
print("Candidate docs:", len(candidate_doc_ids))
print("Method configs:", len(method_registry))
print("doc_embeddings shape:", doc_embeddings.shape)

In [ ]:
# Output paths for the embedding spread experiment
EMBED_SPREAD_DIR = os.path.join(CACHE_DIR, "embedding_spread")
os.makedirs(EMBED_SPREAD_DIR, exist_ok=True)

RETRIEVAL_DOCS_JSONL = os.path.join(EMBED_SPREAD_DIR, "retrieved_docs_by_method.jsonl")
SPREAD_BY_QUERY_CSV = os.path.join(EMBED_SPREAD_DIR, "embedding_spread_by_query.csv")
SPREAD_SUMMARY_CSV = os.path.join(EMBED_SPREAD_DIR, "embedding_spread_method_summary.csv")
SPREAD_REL_BENCHMARK_CSV = os.path.join(EMBED_SPREAD_DIR, "embedding_spread_relevance_benchmarks.csv")

print("EMBED_SPREAD_DIR =", EMBED_SPREAD_DIR)

In [ ]:
import json

def avg_pairwise_cosine_distance_from_doc_ids(doc_ids):
    valid_doc_ids = [d for d in doc_ids if d in doc_index]
    if len(valid_doc_ids) < 2:
        return np.nan
    emb = np.vstack([doc_embeddings[doc_index[d]] for d in valid_doc_ids])
    sim = emb @ emb.T   # normalized embeddings => cosine similarity
    dist = 1.0 - sim
    iu = np.triu_indices(len(valid_doc_ids), k=1)
    vals = dist[iu]
    return float(vals.mean()) if len(vals) else np.nan

def min_pairwise_cosine_distance_from_doc_ids(doc_ids):
    valid_doc_ids = [d for d in doc_ids if d in doc_index]
    if len(valid_doc_ids) < 2:
        return np.nan
    emb = np.vstack([doc_embeddings[doc_index[d]] for d in valid_doc_ids])
    sim = emb @ emb.T
    dist = 1.0 - sim
    iu = np.triu_indices(len(valid_doc_ids), k=1)
    vals = dist[iu]
    return float(vals.min()) if len(vals) else np.nan

def max_pairwise_cosine_distance_from_doc_ids(doc_ids):
    valid_doc_ids = [d for d in doc_ids if d in doc_index]
    if len(valid_doc_ids) < 2:
        return np.nan
    emb = np.vstack([doc_embeddings[doc_index[d]] for d in valid_doc_ids])
    sim = emb @ emb.T
    dist = 1.0 - sim
    iu = np.triu_indices(len(valid_doc_ids), k=1)
    vals = dist[iu]
    return float(vals.max()) if len(vals) else np.nan

def embedding_centroid_radius_from_doc_ids(doc_ids):
    valid_doc_ids = [d for d in doc_ids if d in doc_index]
    if len(valid_doc_ids) < 2:
        return np.nan
    emb = np.vstack([doc_embeddings[doc_index[d]] for d in valid_doc_ids])
    centroid = emb.mean(axis=0)
    norm = np.linalg.norm(centroid)
    if norm > 1e-12:
        centroid = centroid / norm
    sims = emb @ centroid
    dists = 1.0 - sims
    return float(dists.mean())

def relevant_doc_sets(qid):
    judged = qrels_frozen[qid]
    rel_all = [doc_id for doc_id, rel in judged.items() if rel > 0 and doc_id in doc_index]
    rel_g2p = [doc_id for doc_id, rel in judged.items() if rel >= 2 and doc_id in doc_index]
    rel_g3 = [doc_id for doc_id, rel in judged.items() if rel == 3 and doc_id in doc_index]
    return rel_all, rel_g2p, rel_g3

In [ ]:
def build_retrieved_docs_table(method_registry, qids=None, k=TOP_K, out_path=RETRIEVAL_DOCS_JSONL, overwrite=False):
    if qids is None:
        qids = list(queries_frozen.keys())

    if os.path.exists(out_path) and not overwrite:
        rows = []
        with open(out_path, "r") as f:
            for line in f:
                rows.append(json.loads(line))
        return pd.DataFrame(rows)

    rows = []
    for entry in tqdm(method_registry, desc="Methods for spread experiment"):
        method_name = entry["method"]
        family = entry["family"]
        params = entry.get("params", {})
        retriever = entry["retriever"]

        for qid in tqdm(qids, desc=f"Retrieving for {method_name}", leave=False):
            query_text = queries_frozen[qid]
            docs = retriever(query_text, k=k)
            row = {
                "qid": qid,
                "method": method_name,
                "family": family,
                "k": k,
                "retrieved_doc_ids": docs,
            }
            for key, value in params.items():
                row[key] = value
            rows.append(row)

    with open(out_path, "w") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")

    return pd.DataFrame(rows)

retrieved_docs_df = build_retrieved_docs_table(
    method_registry=method_registry,
    qids=full_qids,
    k=TOP_K,
    out_path=RETRIEVAL_DOCS_JSONL,
    overwrite=False,
)

print("retrieved_docs_df shape:", retrieved_docs_df.shape)
retrieved_docs_df.head()

In [ ]:
relevance_benchmark_rows = []

for qid in full_qids:
    rel_all, rel_g2p, rel_g3 = relevant_doc_sets(qid)
    relevance_benchmark_rows.append({
        "qid": qid,
        "num_relevant_docs": len(rel_all),
        "num_grade2plus_docs": len(rel_g2p),
        "num_grade3_docs": len(rel_g3),
        "relevant_avg_pairwise_dist": avg_pairwise_cosine_distance_from_doc_ids(rel_all),
        "relevant_min_pairwise_dist": min_pairwise_cosine_distance_from_doc_ids(rel_all),
        "relevant_max_pairwise_dist": max_pairwise_cosine_distance_from_doc_ids(rel_all),
        "relevant_centroid_radius": embedding_centroid_radius_from_doc_ids(rel_all),
        "grade2plus_avg_pairwise_dist": avg_pairwise_cosine_distance_from_doc_ids(rel_g2p),
        "grade2plus_centroid_radius": embedding_centroid_radius_from_doc_ids(rel_g2p),
        "grade3_avg_pairwise_dist": avg_pairwise_cosine_distance_from_doc_ids(rel_g3),
        "grade3_min_pairwise_dist": min_pairwise_cosine_distance_from_doc_ids(rel_g3),
        "grade3_max_pairwise_dist": max_pairwise_cosine_distance_from_doc_ids(rel_g3),
        "grade3_centroid_radius": embedding_centroid_radius_from_doc_ids(rel_g3),
    })

relevance_benchmarks_df = pd.DataFrame(relevance_benchmark_rows)
relevance_benchmarks_df.to_csv(SPREAD_REL_BENCHMARK_CSV, index=False)

print("Saved:", SPREAD_REL_BENCHMARK_CSV)
relevance_benchmarks_df.head()

In [ ]:
spread_rows = []

for _, row in tqdm(retrieved_docs_df.iterrows(), total=len(retrieved_docs_df), desc="Computing embedding spread stats"):
    qid = row["qid"]
    selected = row["retrieved_doc_ids"][: row["k"]]

    rel_all, rel_g2p, rel_g3 = relevant_doc_sets(qid)

    selected_avg = avg_pairwise_cosine_distance_from_doc_ids(selected)
    selected_min = min_pairwise_cosine_distance_from_doc_ids(selected)
    selected_max = max_pairwise_cosine_distance_from_doc_ids(selected)
    selected_radius = embedding_centroid_radius_from_doc_ids(selected)

    rel_avg = avg_pairwise_cosine_distance_from_doc_ids(rel_all)
    g2p_avg = avg_pairwise_cosine_distance_from_doc_ids(rel_g2p)
    g3_avg = avg_pairwise_cosine_distance_from_doc_ids(rel_g3)

    spread_row = {
        "qid": qid,
        "method": row["method"],
        "family": row["family"],
        "k": row["k"],
        "selected_avg_pairwise_dist": selected_avg,
        "selected_min_pairwise_dist": selected_min,
        "selected_max_pairwise_dist": selected_max,
        "selected_centroid_radius": selected_radius,
        "num_selected": len(selected),
        "num_relevant_docs": len(rel_all),
        "num_grade2plus_docs": len(rel_g2p),
        "num_grade3_docs": len(rel_g3),
        "relevant_avg_pairwise_dist": rel_avg,
        "grade2plus_avg_pairwise_dist": g2p_avg,
        "grade3_avg_pairwise_dist": g3_avg,
        "selected_vs_relevant_ratio": (selected_avg / rel_avg) if pd.notna(selected_avg) and pd.notna(rel_avg) and abs(rel_avg) > 1e-12 else np.nan,
        "selected_vs_grade2plus_ratio": (selected_avg / g2p_avg) if pd.notna(selected_avg) and pd.notna(g2p_avg) and abs(g2p_avg) > 1e-12 else np.nan,
        "selected_vs_grade3_ratio": (selected_avg / g3_avg) if pd.notna(selected_avg) and pd.notna(g3_avg) and abs(g3_avg) > 1e-12 else np.nan,
    }

    for col in ["alpha", "lambda_div", "eta", "sparse_decay", "beta_stop", "min_combined_residual_frac", "min_combined_score"]:
        if col in row.index:
            spread_row[col] = row[col]

    spread_rows.append(spread_row)

spread_df = pd.DataFrame(spread_rows)
spread_df.to_csv(SPREAD_BY_QUERY_CSV, index=False)

print("Saved:", SPREAD_BY_QUERY_CSV)
spread_df.head()

In [ ]:
summary_metric_cols = [
    "selected_avg_pairwise_dist",
    "selected_min_pairwise_dist",
    "selected_max_pairwise_dist",
    "selected_centroid_radius",
    "selected_vs_relevant_ratio",
    "selected_vs_grade2plus_ratio",
    "selected_vs_grade3_ratio",
    "relevant_avg_pairwise_dist",
    "grade2plus_avg_pairwise_dist",
    "grade3_avg_pairwise_dist",
    "num_selected",
    "num_relevant_docs",
    "num_grade2plus_docs",
    "num_grade3_docs",
]

available_summary_metric_cols = [c for c in summary_metric_cols if c in spread_df.columns]
available_param_cols = [c for c in ["alpha", "lambda_div", "eta", "sparse_decay", "beta_stop", "min_combined_residual_frac", "min_combined_score"] if c in spread_df.columns]

spread_summary_df = (
    spread_df.groupby(["method", "family"], as_index=False)[available_summary_metric_cols]
    .mean()
)

if available_param_cols:
    params_df = spread_df[["method", "family"] + available_param_cols].drop_duplicates(subset=["method", "family"])
    spread_summary_df = spread_summary_df.merge(params_df, on=["method", "family"], how="left")

spread_summary_df = spread_summary_df.sort_values("selected_avg_pairwise_dist", ascending=True).reset_index(drop=True)
spread_summary_df.to_csv(SPREAD_SUMMARY_CSV, index=False)

print("Saved:", SPREAD_SUMMARY_CSV)
spread_summary_df.head(20)

In [ ]:
print("Relevant-set benchmarks across queries:")
display(
    relevance_benchmarks_df[[
        "num_relevant_docs",
        "num_grade2plus_docs",
        "num_grade3_docs",
        "relevant_avg_pairwise_dist",
        "grade2plus_avg_pairwise_dist",
        "grade3_avg_pairwise_dist",
    ]].mean().to_frame("mean_across_queries").T.round(4)
)

print("\nMethods with the smallest average selected spread:")
display(
    spread_summary_df[[
        c for c in [
            "method",
            "family",
            "selected_avg_pairwise_dist",
            "selected_vs_relevant_ratio",
            "selected_vs_grade2plus_ratio",
            "selected_vs_grade3_ratio",
            "selected_centroid_radius",
            "relevant_avg_pairwise_dist",
            "grade3_avg_pairwise_dist",
        ] if c in spread_summary_df.columns
    ]].sort_values("selected_avg_pairwise_dist", ascending=True).head(20).round(4)
)

In [ ]:
# Family-level plots
family_spread_plot_df = (
    spread_df.groupby("family", as_index=False)[[
        "selected_avg_pairwise_dist",
        "selected_centroid_radius",
        "selected_vs_relevant_ratio",
        "selected_vs_grade3_ratio",
    ]]
    .mean()
    .sort_values("selected_avg_pairwise_dist", ascending=True)
)

plt.figure(figsize=(10, 5))
plt.bar(family_spread_plot_df["family"], family_spread_plot_df["selected_avg_pairwise_dist"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("Mean selected avg pairwise cosine distance")
plt.title("Embedding-space spread of retrieved docs by family")
plt.grid(axis="y", alpha=0.25)
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(family_spread_plot_df["family"], family_spread_plot_df["selected_vs_relevant_ratio"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("Selected / relevant avg-pairwise-distance ratio")
plt.title("How clustered retrieved docs are relative to all relevant docs")
plt.axhline(1.0, linestyle="--")
plt.grid(axis="y", alpha=0.25)
plt.show()

### How to read the new outputs

The key columns are:

- `selected_avg_pairwise_dist`: mean pairwise cosine distance among the top-k docs selected by a method
- `relevant_avg_pairwise_dist`: mean pairwise cosine distance among all relevant docs for that query
- `grade3_avg_pairwise_dist`: mean pairwise cosine distance among grade-3 docs for that query
- `selected_vs_relevant_ratio`: if this is far below 1, the method retrieves a much tighter cluster than the whole relevant set
- `selected_vs_grade3_ratio`: if this is below 1, the method is even tighter than the highly relevant set

So if dense retrieval has the smallest `selected_avg_pairwise_dist` and the lowest ratios, that is clean evidence that it tends to choose unusually close-together passages in embedding space.